In [1]:
!pip install -q transformers sentencepiece

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

#vocabulario portugues do PTT5
token_name = "unicamp-dl/ptt5-base-portuguese-vocab"
#PTT5 com fine-tuning para resumo
model_name = "recogna-nlp/ptt5-base-summ"

tokenizer = AutoTokenizer.from_pretrained(token_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.eval()

config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/756k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/780 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [6]:
#Função que para retornar o resumo. (Tive que fazer isso porque do jeito do curso tava dando erro)
def resumir(texto, max_length=100, min_length=50, num_beams=4):
    entrada = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        max_length=512,      # limite de contexto do PTT5-base
    )

    with torch.no_grad():
        saida = model.generate(
            **entrada,
            max_length=max_length,
            min_length=min_length,
            num_beams=num_beams,
            no_repeat_ngram_size=3,
            length_penalty=1.0,
            early_stopping=True,
        )

    return tokenizer.decode(saida[0], skip_special_tokens=True)

In [4]:
texto = """Carl Edward Sagan  foi um cientista, físico, biólogo, astrônomo, astrofísico, cosmólogo, escritor, divulgador científico e ativista norte-americano. Sagan é autor de mais de 600 publicações científicas[4][5] e também de mais de vinte livros de ciência e ficção científica.
Foi durante a vida um grande defensor do ceticismo e do uso do método científico. Promoveu a busca por inteligência extraterrestre através do projeto SETI e instituiu o envio de mensagens a bordo de sondas espaciais, destinadas a informar possíveis civilizações extraterrestres sobre a existência humana. Mediante suas observações da atmosfera de Vênus, foi um dos primeiros cientistas a estudar o efeito estufa em escala planetária. Também fundou a organização não governamental Sociedade Planetária e foi pioneiro no ramo da exobiologia. Sagan passou grande parte da carreira como professor da Universidade Cornell, onde foi diretor do laboratório de estudos planetários. Em 1960 obteve o título de doutor pela Universidade de Chicago.
Sagan é conhecido por seus livros de divulgação científica e pela premiada série televisiva de 1980 Cosmos: Uma Viagem Pessoal, que ele mesmo narrou e coescreveu.[9] O livro Cosmos foi publicado para complementar a série. Sagan escreveu o romance Contact, que serviu de base para um filme homônimo de 1997. Em 1978, ganhou o Prémio Pulitzer de Não Ficção Geral pelo seu livro The Dragons of Eden. Morreu aos 62 anos, de pneumonia, depois de uma batalha de dois anos com uma rara e grave doença na medula óssea (mielodisplasia).
Ao longo de sua vida, recebeu vários prêmios e condecorações pelo seu trabalho de divulgação científica. Sagan é considerado um dos divulgadores científicos mais carismáticos e influentes da história, graças a sua capacidade de transmitir as ideias científicas e os aspectos culturais ao público não especializado."""

In [7]:
resumo = resumir(texto, max_length=100, min_length=50)
print(resumo)

Carl Edward Sagan foi um cientista, físico, biólogo, astrônomo, astrofísico, cosmólogo, escritor, divulgador científico e ativista norte-americano. Sagan é conhecido por seus livros de divulgação científica e pela premiada série televisiva de 1980 Cosmos: Uma Viagem Pessoal.
